Libraries
=========

In [2]:
# Start by importing some libraries that may be useful
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import gc

# For this project, we'll use PyTorch instead of TensorFlow
!pip install torch torchvision torchaudio

import torch
import torch.nn as nn

#print(torch.__version__)

In [3]:
print(f"Matplotlib version: {matplotlib.__version__}")

Matplotlib version: 3.10.1


LSTM Class Definition
=====================

In [5]:
# Define a class for LSTM models
class MyLSTM(nn.Module):
    # Initialize LSTM with necessary fields
    def __init__(self, input_dimension, hidden_dimension, layer_dimension, output_dimension, h0, c0):
        super(MyLSTM, self).__init__()
        self.hidden_dimension = hidden_dimension
        self.layer_dimension = layer_dimension
        self.lstm = nn.LSTM(input_dimension, hidden_dimension, layer_dimension, batch_first = True)
        self.fc = nn.Linear(hidden_dimension, output_dimension)
        self.h0 = None
        self.c0 = None
        
    # forward method is used to run data through LSTM model
    def forward(self, x, h0 = None, c0 = None):
        if h0 is None or c0 is None:
            h0 = torch.zeros(self.layer_dimension, x.size(0), self.hidden_dimension).to(x.device)
            c0 = torch.zeros(self.layer_dimension, x.size(0), self.hidden_dimension).to(x.device)

        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out, hn, cn

Model Creation
==============

In [7]:
# Adjust the following values to re-shape the model
num_input_dimensions = 1 # Changes based on input data
num_hidden_dimensions = 10 # Determines how many features are in hidden layers
num_layer_dimensions = 1 # Increase to stack more LSTM's on top of each other

# Now, use the defined class and parameterss to create the model
testModel = MyLSTM(input_dimension = num_input_dimensions,
                   hidden_dimension = num_hidden_dimensions,
                   layer_dimension = num_layer_dimensions,
                   output_dimension = 1,
                   h0 = None,
                   c0 = None)

# Model currently uses Mean Squared Error
# (May adjust criterion as hyper-parameter)
criterion = nn.MSELoss()

# Model also uses ADAM optimizer
# Will most likely use ADAM in final project submit
# Learning rate is adjustable (Default is 0.001)
myLSTM_learning_rate = 0.001
optimizer = torch.optim.Adam(testModel.parameters(), lr = myLSTM_learning_rate)

print("Model Creation Completed!")

Model Creation Completed!


Create Random Dataset
=====================

In [9]:
# Start by creating random dataset
np.random.seed(0)
torch.manual_seed(0)

t = np.linspace (0, 50, 200)
data = np.sin(t)
seq_length = 50

# The below function creates a random tensor usable for the LSTM Model
def generate_random_data(sequence_length):
    xSequence, ySequence = [], []
    
    for i in range(len(data) - sequence_length):
        x = data[i:(i + sequence_length)]
        y = data[i + sequence_length]
        xSequence.append(x)
        ySequence.append(y)

    X = np.array(xSequence)
    y = np.array(ySequence)

    return torch.tensor(X[:, :, None], dtype = torch.float32), torch.tensor(y[:, None], dtype = torch.float32)

# Generate X_train and y_train using the generate_random_data() function
X_train, y_train = generate_random_data(seq_length)
#print(X_train)

Train Model
===========

In [11]:
# model_train method takes in a model, number of epochs, and a number of epochs before displaying progress
def model_train(model, num_epochs = 100, epoch_display_length = 10):
    h0, c0 = None, None
    
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()

        outputs, h0, c0 = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

        # Clear memory between iterations
        if hasattr(torch.cuda, 'empty_cache'):
            torch.cuda.empty_cache()

        if(epoch + 1) % epoch_display_length == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# Train testModel using model_train() method
model_train(model = testModel, num_epochs = 50, epoch_display_length = 10)

Epoch [10/50], Loss: 0.6114
Epoch [20/50], Loss: 0.5756
Epoch [30/50], Loss: 0.5439
Epoch [40/50], Loss: 0.5141
Epoch [50/50], Loss: 0.4848


Model Predictions
=================

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

testModel.eval()

with torch.no_grad():
    predicted, _, _ = testModel(X_train)

predicted_np = predicted.detach().cpu().numpy()
original_values = data[seq_length:]
time_steps = np.arange(seq_length, len(data))

#predicted[::30] += 0.2
#predicted[::70] -= 0.2

# Clear cache, and close any existing plots, if applicable
if torch.cuda.is_available():
    torch.cuda.empty_cache()
del predicted
gc.collect()
plt.close('all')

# Define plot features
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(time_steps, original_values, label='Original Data')
ax.plot(time_steps, predicted_np, label='Prediction Data', linestyle='--')
ax.set_title('LSTM Predictions compared to Original Data')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.legend()

plt.tight_layout()
plt.show()



print("Test Complete")